# Capability Host Troubleshooting for AI Foundry (Project-Based)

This notebook helps you manage Azure AI Foundry capability hosts for network-secured agent deployments.

## ⚠️ IMPORTANT: Execute Cell-by-Cell

**DO NOT use "Run All"!** This notebook contains operations that:
- Create Azure resources (costs money)
- Delete Azure resources (permanent)
- Require manual configuration before running

**Always run cells one by one** and read the comments in each cell before executing.

## 📚 Quick Start for Read-Only Exploration

If you just want to check the status of your capability hosts without making changes:

1. **Run Cell 1**: Install dependencies
2. **Run Cell 2**: Setup and authentication
3. **Run Cell 3**: Get Account Capability Host (read-only)
4. **Run Cell 4**: Get Project Capability Host (read-only)

These cells are safe to run and won't modify your infrastructure.

In [ ]:
# Install required packages for Azure authentication and API calls
# This cell is safe to run multiple times

!pip install requests azure.identity python-dotenv

## Prerequisites

Before running this notebook, ensure you have:

1. **Azure Subscription** with an AI Foundry project deployed
2. **Azure CLI** installed and logged in (`az login`)
3. **Python 3.7+** with pip
4. **Permissions**: Contributor or higher on the AI Foundry project
5. **`.env` file**: Created from `.env.example` with your values filled in

### Required Environment Variables

Create a `.env` file in the `utils/` directory with:

```bash
projectResourceId=/subscriptions/{subscriptionId}/resourceGroups/{rg}/providers/Microsoft.CognitiveServices/accounts/{foundryName}/projects/{projectName}
subnetId=/subscriptions/{subscriptionId}/resourceGroups/{rg}/providers/Microsoft.Network/virtualNetworks/{vnetName}/subnets/{subnetName}
tenantId={your-tenant-id}  # Optional, for multi-tenant scenarios
```

In [ ]:
# ========================================
# SETUP: Load Environment and Authenticate
# ========================================
# This cell:
# 1. Loads configuration from .env file
# 2. Parses the project resource ID to extract Azure resource details
# 3. Authenticates with Azure using DefaultAzureCredential
# 4. Prepares headers for API calls
#
# Safe to run: Yes (read-only operations)
# ========================================

import requests
import json
import os
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(verbose=True, override=True)

# Get project resource ID from environment (REQUIRED)
project_resource_id = os.environ.get("projectResourceId", None)

# Get subnet ID from environment (OPTIONAL - only needed for creating capability hosts)
subnet_id = os.environ.get("subnetId", None)

# Validate required environment variable
if project_resource_id is None:
    raise ValueError("❌ projectResourceId environment variable is not set. 🔧 Please set it in your .env file.")

# Parse project_resource_id to extract individual components
# Format: /subscriptions/{subscriptionId}/resourceGroups/{rg}/providers/Microsoft.CognitiveServices/accounts/{foundryName}/projects/{projectName}
parts = project_resource_id.split("/")
try:
    subscription_id = parts[parts.index("subscriptions") + 1]  # Azure subscription ID
    rg = parts[parts.index("resourceGroups") + 1]              # Resource group name
    foundry_name = parts[parts.index("accounts") + 1]          # AI Foundry account name
    project_name = parts[parts.index("projects") + 1]          # AI Foundry project name
except (ValueError, IndexError) as e:
    raise ValueError("❌ Invalid projectResourceId format. 📝 Expected format: /subscriptions/{{subscriptionId}}/resourceGroups/{{rg}}/providers/Microsoft.CognitiveServices/accounts/{{foundryName}}/projects/{{projectName}}") from e

# Display parsed configuration
print(f"✅ Parsed project resource ID: {project_resource_id[:50]}...")
print(f"   📦 Subscription: {subscription_id}")
print(f"   📁 Resource Group: {rg}")
print(f"   🏭 Foundry: {foundry_name}")
print(f"   📋 Project: {project_name}")

# Display helpful Azure CLI commands for reference
print("\n🔧 Azure CLI commands to get capability hosts:")
print("   Get capability hosts:")
print(f"     🔧 az rest --method get --url https://management.azure.com/subscriptions/{subscription_id}/resourceGroups/{rg}/providers/Microsoft.CognitiveServices/accounts/{foundry_name}/projects/{project_name}/capabilityHosts?api-version=2025-04-01-preview")
print("   Delete account capability host:")
print(f"     🔧 az rest --method delete --url https://management.azure.com/subscriptions/{subscription_id}/resourceGroups/{rg}/providers/Microsoft.CognitiveServices/accounts/{foundry_name}/projects/{project_name}/capabilityHosts/{foundry_name}@aml_aiagentservice?api-version=2025-04-01-preview")
print("   Create account capability host:")
print(f"     🔧 az rest --method put --url https://management.azure.com/subscriptions/{subscription_id}/resourceGroups/{rg}/providers/Microsoft.CognitiveServices/accounts/{foundry_name}/projects/{project_name}/capabilityHosts/{foundry_name}caphost?api-version=2025-04-01-preview --body '{{\"properties\": {{\"customerSubnet\": \"{subnet_id}\", \"capabilityHostKind\": \"Agents\"}}}}'")
print("   Get project capability hosts:")
print(f"     🔧 az rest --method get --url https://management.azure.com/subscriptions/{subscription_id}/resourceGroups/{rg}/providers/Microsoft.CognitiveServices/accounts/{foundry_name}/projects/{project_name}/capabilityHosts?api-version=2025-04-01-preview")

# Get tenant ID (optional, for multi-tenant scenarios)
tenant_id = os.environ.get("tenantId", None)

# Construct base URL for API calls
base_url = f"https://management.azure.com/subscriptions/{subscription_id}/resourceGroups/{rg}/providers/Microsoft.CognitiveServices/accounts/"

# Authenticate with Azure using DefaultAzureCredential
# This will try multiple authentication methods in order:
# 1. Environment variables
# 2. Managed Identity
# 3. Azure CLI
# 4. Visual Studio Code
# 5. Azure PowerShell
credential = DefaultAzureCredential()
token = credential.get_token(
    "https://management.azure.com/.default", tenant_id=tenant_id
)

# Prepare HTTP headers for API calls
headers = {"Authorization": f"Bearer {token.token}", "Content-Type": "application/json"}

print("\n✅ Setup complete! Ready to execute operations.")

## 🎯 What Should I Do?

Choose your scenario below and follow the recommended cell execution order:

### 📊 First Time Setup?
**Goal**: Set up capability hosts for a new AI Foundry project

**Steps**:
1. Run: **Get Account Capability Host** (check if it exists)
2. Run: **Get Project Capability Host** (check if it exists)
3. If not found: Run **Create Account Capability Host** (after editing subnet ID)
4. If not found: Run **Create Project Capability Host** (after editing connection names)
5. Run: **POST to Azure OpenAI Service** (validate connectivity)

### 🔧 Troubleshooting Failed Deployment?
**Goal**: Recover from a capability host in "Failed" state

**Steps**:
1. Run: **Get Account Capability Host** → Check for "Failed" in provisioningState
2. Run: **Get Project Capability Host** → Check for "Failed" in provisioningState
3. If Failed: Run **Delete Project Capability Host** first (must delete project before account)
4. If Failed: Run **Delete Account Capability Host**
5. Wait 5 minutes for deletion to complete
6. Run: **Create Account Capability Host** (after editing subnet ID)
7. Run: **Create Project Capability Host** (after editing connection names)

### 👀 Just Checking Status?
**Goal**: View current capability host configuration (read-only)

**Steps**:
1. Run: **Get Account Capability Host** ✅ Safe
2. Run: **Get Project Capability Host** ✅ Safe
3. Run: **Get Connections** ✅ Safe

### 🧪 Testing Connectivity?
**Goal**: Verify your AI Foundry setup works end-to-end

**Steps**:
1. Run: **POST to Azure OpenAI Service** (sends test prompt)
2. Run: **Get Agents** (lists configured agents)

---

**⚠️ Important Notes**:
- Always check current state with GET operations before making changes
- Delete operations are **permanent** - use only for recovery
- Create operations may take 10-15 minutes to complete
- If a capability host is "Provisioning", wait for it to complete before taking action

---

# 📊 Diagnostic Operations (Read-Only)

The following cells are **safe to run** as they only query existing resources without making modifications.

### Get Account Capability Host

In [ ]:
# ========================================
# GET ACCOUNT CAPABILITY HOST
# ========================================
# This cell retrieves the account-level capability host configuration.
# 
# Account capability host:
# - Provides VNET integration for the AI Foundry account
# - Defines the subnet used for agent execution
# - Must be created before project capability host
#
# What to check in the output:
# - "provisioningState": Should be "Succeeded" (not "Failed" or "Provisioning")
# - "customerSubnet": Should match your expected subnet ID
# - "capabilityHostKind": Should be "Agents"
#
# Safe to run: Yes (read-only)
# ========================================

url = f"{base_url}{foundry_name}/capabilityHosts/?api-version=2025-06-01"
response = requests.get(url, headers=headers)
response_json = response.json()

# Check if a subnet is configured (indicates VNET injection is enabled)
found_subnet = response_json['value'][0]['properties']['customerSubnet'] if 'value' in response_json and len(response_json['value']) > 0 else None
if found_subnet:
    print(f"✅ Foundry using subnet: {found_subnet}")
else:
    print("❌ No subnet found. Foundry is NOT VNET INJECTED.")
    
print(json.dumps(response_json, indent=4))

### Get Project Capability Host

In [ ]:
# ========================================
# GET PROJECT CAPABILITY HOST
# ========================================
# This cell retrieves the project-level capability host configuration.
#
# Project capability host:
# - Links project-specific connections (Storage, Cosmos DB, AI Search)
# - Required for agents to access your data sources
# - Depends on account capability host being created first
#
# What to check in the output:
# - "provisioningState": Should be "Succeeded"
# - "vectorStoreConnections": AI Search connection names
# - "storageConnections": Storage account connection names
# - "threadStorageConnections": Cosmos DB connection names
#
# Safe to run: Yes (read-only)
# ========================================

url = f"{base_url}{foundry_name}/projects/{project_name}/capabilityHosts?api-version=2025-06-01"
print(url)  # For debugging purposes
response = requests.get(url, headers=headers)
print(response.status_code)
print(json.dumps(response.json(), indent=4))

---

# ⚠️ Modification Operations (Use with Caution)

The following cells will **create or delete Azure resources**. Read all warnings and edit placeholder values before running.

### Create Account Capability Host

In [ ]:
# ========================================
# CREATE ACCOUNT CAPABILITY HOST
# ========================================
# ⚠️ WARNING: This cell creates an Azure resource!
#
# BEFORE RUNNING:
# 1. Replace "<subnet_resource_id>" below with your actual subnet resource ID
# 2. Find your subnet ID: Azure Portal → Virtual Network → Subnets → Properties → Resource ID
# 3. Ensure the subnet is in the SAME REGION as your AI Foundry account
# 4. Ensure the subnet has sufficient address space (at least /28)
#
# This operation:
# - Creates VNET injection for your AI Foundry account
# - Takes 10-15 minutes to complete
# - Cannot be modified once created (must delete and recreate)
#
# Safe to run: NO - creates a resource
# ========================================

# VALIDATION: Check if subnet_id is still a placeholder
placeholder_subnet = "<subnet_resource_id>"
actual_subnet_id = subnet_id if subnet_id else placeholder_subnet

if actual_subnet_id == placeholder_subnet or actual_subnet_id is None:
    print("❌ ERROR: subnet_id is not configured!")
    print("")
    print("Please do ONE of the following:")
    print("  1. Set 'subnetId' in your .env file, then re-run the Setup cell")
    print("  2. OR edit the 'customerSubnet' value in the payload below")
    print("")
    print("Your subnet ID should look like:")
    print("/subscriptions/{subscriptionId}/resourceGroups/{rg}/providers/Microsoft.Network/virtualNetworks/{vnetName}/subnets/{subnetName}")
    print("")
    print("⚠️ Exiting to prevent creating capability host with invalid subnet.")
else:
    print(f"✅ Using subnet: {actual_subnet_id}")
    print("")
    
    url = f"{base_url}{foundry_name}/capabilityHosts/{foundry_name}caphost?api-version=2025-06-01"
    payload = {
        "properties": {
            "capabilityHostKind": "Agents",
            "customerSubnet": actual_subnet_id,  # ⚠️ EDIT THIS if not using .env
        }
    }
    
    print("Creating account capability host...")
    print(f"API URL: {url}")
    print("Payload:")
    print(json.dumps(payload, indent=4))
    
    response = requests.put(url, headers=headers, data=json.dumps(payload))
    print("\nResponse:")
    print(json.dumps(response.json(), indent=4))

### Create Project Capability Host

In [ ]:
# ========================================
# CREATE PROJECT CAPABILITY HOST
# ========================================
# ⚠️ WARNING: This cell creates an Azure resource!
#
# BEFORE RUNNING:
# 1. Verify the connection names in the payload below exist in your project
# 2. Run the "Get Connections" cell (in Connections section) to see available connections
# 3. Update the connection names in the payload below to match YOUR project's connections
#
# Connection types required:
# - vectorStoreConnections: AI Search connection(s)
# - storageConnections: Storage account connection(s)
# - threadStorageConnections: Cosmos DB connection(s)
#
# This operation:
# - Links your project to required data sources
# - Takes 5-10 minutes to complete
# - Requires account capability host to exist first
#
# Safe to run: NO - creates a resource
# ========================================

# ⚠️ IMPORTANT: Edit these connection names to match YOUR project!
# The values below are examples and will NOT work for your project.
# Run the "Get Connections" cell to find your actual connection names.
vector_store_connections = ["srch-mfgai-p-naa-007-for-ai-project-1"]  # ⚠️ EDIT THIS
storage_connections = ["stmfgaipnaa007-for-ai-project-1"]            # ⚠️ EDIT THIS
thread_storage_connections = ["cosmos-mfgai-p-naa-007-for-ai-project-1"]  # ⚠️ EDIT THIS

# Warn if connection names look like defaults
if "mfgai" in str(vector_store_connections).lower() or "007" in str(vector_store_connections):
    print("⚠️ WARNING: Connection names appear to be default values!")
    print("Please verify these are YOUR actual connection names.")
    print("Run the 'Get Connections' cell to see available connections.")
    print("")

url = f"{base_url}{foundry_name}/projects/{project_name}/capabilityHosts/projcaphost?api-version=2025-06-01"
payload = {
    "properties": {
        "capabilityHostKind": "Agents",
        "vectorStoreConnections": vector_store_connections,
        "storageConnections": storage_connections,
        "threadStorageConnections": thread_storage_connections,
    }
}

print("Creating project capability host...")
print(f"API URL: {url}")
print("Payload:")
print(json.dumps(payload, indent=4))

response = requests.put(url, headers=headers, data=json.dumps(payload))
print("\nResponse:")
print(json.dumps(response.json(), indent=4))

### Delete Account Capability Host

In [ ]:
# ========================================
# 🚨 DELETE ACCOUNT CAPABILITY HOST 🚨
# ========================================
# ⛔ DANGER: This cell PERMANENTLY DELETES an Azure resource!
#
# WHEN TO USE:
# ✅ Capability host is in "Failed" provisioning state
# ✅ You need to change the subnet (requires delete + recreate)
# ✅ You're decommissioning the VNET integration
#
# WHEN NOT TO USE:
# ❌ Capability host is "Provisioning" (wait for it to complete)
# ❌ Capability host is "Succeeded" (no need to delete)
# ❌ You haven't checked the current state first
#
# BEFORE RUNNING:
# 1. Run "Get Account Capability Host" to check current state
# 2. Ensure project capability host is deleted first (if it exists)
# 3. Understand that deletion is permanent and takes 5-10 minutes
#
# Safe to run: NO - deletes a resource permanently
# ========================================

print("🚨 DANGER: You are about to DELETE the account capability host!")
print("")
print("This will:")
print("  - Remove VNET integration from your AI Foundry account")
print("  - Require recreation if you need it again")
print("  - Take 5-10 minutes to complete")
print("")
print("⚠️ Before proceeding, verify:")
print("  1. You've deleted the project capability host first")
print("  2. The capability host is in 'Failed' state (check with GET first)")
print("  3. You really need to delete it (not just recreate)")
print("")

url = f"{base_url}{foundry_name}/capabilityHosts/{foundry_name}@aml_aiagentservice?api-version=2025-06-01"

# Require explicit user confirmation
user_input = input("Type YES (all caps) to continue with deletion or Ctrl+C to abort: ")
if user_input == "YES":
    print("\nDeleting account capability host...")
    response = requests.delete(url, headers=headers)
    print(f"Response status: {response.status_code}")
    if response.status_code == 202:
        print("✅ Deletion initiated. Wait 5-10 minutes for it to complete.")
    elif response.status_code == 204:
        print("✅ Deletion completed successfully.")
    else:
        print("❌ Deletion failed. Check the error message below:")
        print(response.text)
else:
    print("❌ Deletion cancelled. You must type 'YES' exactly to proceed.")

### Delete Project Capability Host

In [ ]:
# ========================================
# 🚨 DELETE PROJECT CAPABILITY HOST 🚨
# ========================================
# ⛔ DANGER: This cell PERMANENTLY DELETES an Azure resource!
#
# WHEN TO USE:
# ✅ Capability host is in "Failed" provisioning state
# ✅ You need to change connection configurations
# ✅ You're decommissioning the project's agent capability
#
# IMPORTANT:
# - Must be deleted BEFORE deleting account capability host
# - Deletion takes 5-10 minutes
# - Check current state first with "Get Project Capability Host"
#
# Safe to run: NO - deletes a resource permanently
# ========================================

print("🚨 DANGER: You are about to DELETE the project capability host!")
print("")
print("This will:")
print("  - Remove agent capability from your project")
print("  - Require recreation if you need it again")
print("  - Take 5-10 minutes to complete")
print("")

url = f"{base_url}{foundry_name}/projects/{project_name}/capabilityHosts/projcaphost?api-version=2025-06-01"

# Require explicit user confirmation
user_input = input("Type YES (all caps) to continue with deletion or Ctrl+C to abort: ")
if user_input == "YES":
    print("\nDeleting project capability host...")
    response = requests.delete(url, headers=headers)
    print(f"Response status: {response.status_code}")
    if response.status_code == 202:
        print("✅ Deletion initiated. Wait 5-10 minutes for it to complete.")
    elif response.status_code == 204:
        print("✅ Deletion completed successfully.")
    else:
        print("❌ Deletion failed. Check the error message below:")
        print(response.text)
else:
    print("❌ Deletion cancelled. You must type 'YES' exactly to proceed.")

---

# 🧪 Validation & Testing

These cells help you verify that your AI Foundry setup is working correctly.

### POST to Azure OpenAI Service

In [ ]:
# ========================================
# TEST: POST TO AZURE OPENAI SERVICE
# ========================================
# This cell tests end-to-end connectivity by:
# 1. Getting the list of model deployments
# 2. Sending a test chat completion request
#
# What this validates:
# - Authentication is working
# - Your project has model deployments
# - Network connectivity is functioning
# - AI Foundry service is responsive
#
# Safe to run: Yes (sends one test request)
# ========================================

# Get a token for AI Foundry service endpoint
ai_token = credential.get_token("https://ai.azure.com")
headers = {
    "Content-Type": "application/json",
    "Authorization": "Bearer " + ai_token.token,
}

# Get available model deployments
print("Fetching available model deployments...")
url = f"https://{foundry_name}.services.ai.azure.com/api/projects/{project_name}/deployments?api-version=v1"
response = requests.get(url, headers=headers)

# Select the first available deployment
deployment_name = response.json().get("value", [{}])[0].get("name", None)
if deployment_name:
    print(f"✅ Using deployment: {deployment_name}")
else:
    print("❌ No model deployments found. Please deploy a model first.")
    print("Available deployments:")
    print(json.dumps(response.json(), indent=4))

# Send a test chat completion request
if deployment_name:
    print("\nSending test chat completion request...")
    url = f"https://{foundry_name}.services.ai.azure.com/models/chat/completions?api-version=2024-05-01-preview"
    
    payload = {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Does Azure OpenAI support customer managed keys?"},
            {
                "role": "assistant",
                "content": "Yes, customer managed keys are supported by Azure OpenAI.",
            },
            {"role": "user", "content": "Do other Azure AI services support this too?"},
        ],
        "model": deployment_name,
    }
    
    response = requests.post(url, headers=headers, data=json.dumps(payload))
    print("\nResponse:")
    print(json.dumps(response.json(), indent=4))

---

# 🔗 Connections

View and manage project connections (Storage, AI Search, Cosmos DB, etc.)

### Get Connections

In [ ]:
# ========================================
# GET CONNECTIONS
# ========================================
# This cell lists all connections configured in your project.
#
# Connection types you'll see:
# - AISearch: Vector store connections
# - AzureBlob: Storage connections
# - CosmosDB: Thread storage connections
# - ModelGateway: External model connections
#
# Use this to find the correct connection names when creating
# a project capability host.
#
# Safe to run: Yes (read-only)
# ========================================

url = f"{base_url}{foundry_name}/projects/{project_name}/connections?api-version=2025-06-01"
print(url)  # For debugging purposes
response = requests.get(url, headers=headers)
response_json = response.json()

print("\n📋 Available Connections:\n")
for connection in response_json.get("value", []):
    print(f"  Name: {connection.get('name')}")
    print(f"  Category: {connection.get('properties', {}).get('category')}")
    print(f"  ID: {connection.get('id')}")
    print("-" * 60)

print("\n\n📄 Full JSON Response:\n")
print(json.dumps(response_json, indent=4))

### Update Connection (Example)

In [ ]:
# ========================================
# UPDATE CONNECTION (EXAMPLE)
# ========================================
# ⚠️ WARNING: This is an example cell for updating a connection.
# 
# BEFORE RUNNING:
# 1. Replace 'connection_name' with your actual connection name
# 2. Update the payload to match your connection's configuration
# 3. Ensure you have the correct API key or credentials
#
# This cell is provided as a template and may need significant
# customization for your specific use case.
#
# Safe to run: NO - modifies a connection
# ========================================

connection_name = "model-gateway-bro6ttlxdau5e-static"  # ⚠️ EDIT THIS

print(f"⚠️ You are about to update connection: {connection_name}")
print("Ensure you have the correct connection name and configuration.\n")

url = f"{base_url}{foundry_name}/projects/{project_name}/connections/{connection_name}?api-version=2025-06-01"
key = input(f"Enter the API key for the connection {connection_name}: ")

payload = {
    "name": connection_name,
    "type": "Microsoft.CognitiveServices/accounts/projects/connections",
    "properties": {
        "authType": "ApiKey",
        "credentials": {
            "key": key
        },
        "group": "AzureAI",
        "category": "ModelGateway",
        "target": "https://apim-bro6ttlxdau5e.azure-api.net/inference/openai",
        "isSharedToAll": True,
        "metadata": {
            "deploymentInPath": "false",
            "inferenceAPIVersion": "2025-03-01-preview",
            "models": "[{\"name\":\"gpt-4.1-mini\",\"properties\":{\"model\":{\"name\":\"gpt-4.1-mini\",\"version\":\"2025-01-01-preview\",\"format\":\"OpenAI\"}}},{\"name\":\"gpt-5-mini\",\"properties\":{\"model\":{\"name\":\"gpt-5-mini\",\"version\":\"2025-04-01-preview\",\"format\":\"OpenAI\"}}},{\"name\":\"o3-mini\",\"properties\":{\"model\":{\"name\":\"o3-mini\",\"version\":\"2025-01-01-preview\",\"format\":\"OpenAI\"}}}]"
        }
    }
}

response = requests.put(url, headers=headers, data=json.dumps(payload))
print(json.dumps(response.json(), indent=4))

---

# 🤖 Agents

View configured AI agents in your project.

### Get Agents

In [ ]:
# ========================================
# GET AGENTS
# ========================================
# This cell retrieves all agents (assistants) configured in your project.
#
# Agents are AI assistants that can:
# - Use tools (code interpreter, file search, function calling)
# - Access your vector stores and knowledge bases
# - Execute in your VNET-injected environment
#
# Safe to run: Yes (read-only)
# ========================================

url = f"https://{foundry_name}.services.ai.azure.com/api/projects/{project_name}/assistants?api-version=v1"
print(url)  # For debugging purposes

# Get token for AI Foundry service
token = credential.get_token("https://ai.azure.com/.default", tenant_id=tenant_id)
headers = {"Authorization": f"Bearer {token.token}"}

response = requests.get(url, headers=headers)
response_json = response.json()
print(json.dumps(response_json, indent=4))